In [ ]:
pip install gptqmodel

In [5]:
!nvidia-smi

Sat May 16 09:00:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123 \
  --force-reinstall --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB ? eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gptqmodel 7.0.0 requires numpy==2.2.6; python_version < "3.14", but you have numpy 2.4.5 which is incompatible.
google-cloud-aiplatform 1.148.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.5 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.34.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.34.1 which is incompatible.


In [2]:
!pip install huggingface-hub --quiet

<span style="color:#4338CA; font-size:22px; font-family:Trebuchet MS;"><b>📌 Important Note</b></span>

> In this notebook, we will explore quantization from a high-level perspective. After the hackathon, you are encouraged to study the concept in greater depth and understand its low-level implementation details.
>
> To make the learning process easier and more intuitive, most explanations will be supported with visual illustrations and diagrams. If you encounter any difficulties or unclear concepts, do not hesitate to ask a mentor for assistance.


# What is Quantization

> When selecting a specific Large Language Model (LLM) to run, the first question to ask is:
>
> **How much RAM or VRAM is required?**

<center>
<img src="https://i.ibb.co/zVVVbzs9/Fig-Jam-Untitled-12.png" alt="Fig Jam Untitled (12)" border="0">
</center>

> To estimate the required memory, two main factors must be considered:
> * The **number of model parameters**
>* The **numerical precision** used (such as FP32, FP16, INT8, or INT4)
>
>The following equation can be used to estimate the approximate memory consumption of a model:

<center>
<img src="https://i.ibb.co/wr7n8kmj/Untitled-Fig-Jam-8.png" alt="Untitled Fig Jam (8)" border="0">
</center>

> In many real-world scenarios, the available hardware memory may not be sufficient to load or run the model efficiently.
>
> To address this limitation, **quantization** techniques are used.

<center>
<img src="https://i.ibb.co/CpTzspYh/Fig-Jam-Untitled-9.png" alt="Fig Jam Untitled (9)" border="0">
</center>

> Quantization reduces the precision of model weights and activations, which significantly decreases memory usage and can improve inference speed while maintaining acceptable model performance.
>
>There are several quantization techniques available. The following diagram summarizes some of the most important approaches:

<center>
<img src="https://i.ibb.co/hJd4GC2f/Fig-Jam-Untitled-10.png" alt="Fig Jam Untitled (10)" border="0">
</center>

>When we are using quantization, we start with a high-quality, high-precision model and then convert it into a lower-precision version. This results in different trade-offs in memory usage, speed, and output quality depending on the quantization technique used.

>As shown in the diagram, reducing precision significantly decreases the memory required to store and run the model, which makes it possible to deploy large models on limited hardware.

>Another important aspect is **time efficiency**. In general, **Post-Training Quantization (PTQ)** is faster because it is applied directly to a fully trained model without requiring additional optimization steps. On the other hand, **AWQ (Activation-aware Weight Quantization)** typically takes more time, since it carefully analyzes activations and uses a more refined calibration process to preserve accuracy.


<center>
<img src="https://i.ibb.co/5zrKgZ9/Untitled-Fig-Jam-10.png" alt="Untitled Fig Jam (10)" border="0" height="500">
</center>

# 1. AWQ: Activation-aware Weight Quantization

#### Load AWQ Quantized Model

* please see the model card that's we will run [qwen-model-awq](https://huggingface.co/Qwen/Qwen3-8B-AWQ)

If you want to use an already quantized **AWQ** model from <b>Hugging Face</b>, you will notice that most repositories include the term <span style="color:#7C3AED;"><b>“AWQ”</b></span> in the model name.
<center>
<img src="https://i.ibb.co/ksPBsgVK/Untitled-Fig-Jam-7.png" alt="Untitled Fig Jam (7)" border="0">
</center>


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, AwqConfig

model_id = "Qwen/Qwen3-8B-AWQ" # model you want to load
tokenizer = AutoTokenizer.from_pretrained(model_id)
quantization_config = AwqConfig(
    bits=4,
    do_fuse=False,
    exllama_config=None
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config,
    trust_remote_code=True
)

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.However, loading attributes (e.g. ['backend', 'max_input_length']) will be overwritten with the one you passed to `from_pretrained`. The rest will be ignored.
  warnings.warn(warning_msg)


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


...

INFO  Kernel: Auto-selection: adding candidate `AwqExllamaV2Linear`            


INFO  Kernel: selected -> `AwqExllamaV2Linear`.                                


Loading weights:   0%|          | 0/903 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

INFO  gc.collect() reclaimed 8009 objects in 0.357s                            


In [2]:
from transformers import AutoTokenizer, pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
result = pipe([
    {"role":"user","content":"Hello"}
])
response = result[0]["generated_text"][-1]["content"]
print(response)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<think>
Okay, the user said "Hello". That's a greeting. I should respond politely. Let me check the guidelines. I need to be friendly and offer help. Maybe say hello back and ask how I can assist. Keep it simple and open-ended. Make sure it's in the same language as the query, which is English here. Alright, that should work.
</think>

Hello! How can I assist you today? 😊


### [RUN: OPTIONAL] Model Quantization with AWQ Application

**-> in this part because it will took time just take a look in the output without runing the cell code.**

* To do quantize any model using AWQ technique use [llmcompressor](https://github.com/vllm-project/llm-compressor)

In [ ]:
!uv pip install llmcompressor

Using Python 3.12.13 environment at: /usr
Resolved 75 packages in 989ms
Prepared 8 packages in 978ms
Uninstalled 4 packages in 174ms
Installed 8 packages in 129ms
 - accelerate==1.13.0
 + accelerate==1.12.0
 + auto-round==0.10.2
 + compressed-tensors==0.14.0.1
 - huggingface-hub==1.11.0
 + huggingface-hub==0.36.2
 + llmcompressor==0.10.0.2
 + loguru==0.7.3
 - nvidia-ml-py==13.595.45
 + nvidia-ml-py==13.590.48
 - transformers==5.8.1
 + transformers==4.57.6


In [ ]:
from llmcompressor.modifiers.quantization import QuantizationModifier
from llmcompressor.modifiers.awq import AWQModifier

In [ ]:
recipe = [
    AWQModifier(),
    QuantizationModifier(
        ignore=["lm_head"], #
        scheme="W4A16_ASYM",
        targets=["Linear"]
    )
]

In [ ]:
model_path = "Qwen/Qwen3-1.7B"
quant_path  = "./Qwen3-1.7B"   # Where to save the quantized model

In [ ]:
from llmcompressor import oneshot

oneshot(
    model=model_path,
    recipe=recipe,
    dataset="open_platypus",  # Built-in calibration dataset
    output_dir=quant_path,
    max_seq_length=2048,
    num_calibration_samples=128  # Adjust based on your GPU memory
)

2026-05-14T23:50:02.504544+0000 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Preprocessing:   0%|          | 0/24926 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/24926 [00:00<?, ? examples/s]

2026-05-14T23:52:31.044108+0000 | reset | INFO - Compression lifecycle reset
2026-05-14T23:52:31.046597+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-05-14T23:52:31.051784+0000 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...
2026-05-14T23:52:31.062444+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.0.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$'], activation_hook_target=None) because found incompatible balance layers
2026-05-14T23:52:31.063798+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.1.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$'], activation_hook_target=None) because found incompatible balance layers
2026-05-14T23:52:31.065289+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.2.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', b

W0514 23:52:33.985000 5047 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
Smoothing:   0%|          | 0/3 [01:19<?, ?it/s]


KeyboardInterrupt: 

# GPTQ: Generalized Post-Training Quantization

#### Load GPTQ Quantized Model

<center>
<img src="https://i.ibb.co/1tx0hrfq/Fig-Jam-Untitled-11.png" alt="Fig Jam Untitled (11)" border="0">
</center>

In [3]:
! uv pip install unsloth

Using Python 3.12.13 environment at: /usr
Resolved 101 packages in 923ms
Prepared 11 packages in 2.93s
Uninstalled 2 packages in 350ms
Installed 11 packages in 147ms
 + bitsandbytes==0.49.2
 + cut-cross-entropy==25.1.1
 - datasets==4.0.0
 + datasets==4.3.0
 + hf-transfer==0.1.9
 + msgspec==0.21.1
 - transformers==5.8.1
 + transformers==5.5.0
 + trl==0.24.0
 + tyro==1.0.13
 + unsloth==2026.5.2
 + unsloth-zoo==2026.5.1
 + xformers==0.0.35


In [11]:

from unsloth import FastLanguageModel
import torch

model_id = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.8.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536, padding_idx=151665)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RM

In [5]:
from transformers import pipeline
pipe = pipeline("text-generation",model=model,tokenizer=tokenizer)
response = pipe(
    text_inputs=[
        {"role":"user","content":"Hello from me"}
    ]
)[-1]["generated_text"][-1]["content"]
print(response)

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'cache_implementation'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, Future

Hello! How can I assist you today? If you have any questions or need help with something, feel free to ask.


### [RUN: OPTIONAL] Model Quantization with GPTQ Application

* we will use [**GPTQModel**](https://github.com/modelcloud/gptqmodel) package to do quantization

In [ ]:
! uv pip install gptqmodel

Using Python 3.12.13 environment at: /usr
Resolved 87 packages in 644ms
Prepared 10 packages in 9.60s
Uninstalled 2 packages in 35ms
Installed 10 packages in 43ms
 + defuser==0.0.21
 + device-smi==0.5.6
 + gptqmodel==7.0.0
 + logbar==0.4.3
 + maturin==1.13.3
 + ninja==1.13.0
 - numpy==2.0.2
 + numpy==2.2.6
 - protobuf==5.29.6
 + protobuf==7.34.1
 + pypcre==0.3.2
 + tokenicer==0.0.13


In [ ]:
#from datasets import load_dataset
from gptqmodel import GPTQModel, QuantizeConfig

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.0.0
Transformers : 5.5.0
Torch        : 2.10.0+cu128
Triton       : 3.6.0


In [ ]:
model_id = "liquidai/lfm2.5-350m"
quant_path = "lfm2.5-350m-gptqmodel-4bit"

In [ ]:
quant_config = QuantizeConfig(bits=4, group_size=128)

model = GPTQModel.load(model_id, quant_config)

INFO  QuantizeConfig: offload_to_disk_path auto set to temporary dir `/tmp/gptqmodel_iouo32iw`


config.json: 0.00B [00:00, ?B/s]

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

INFO  Loader: Auto dtype (native bfloat16): `torch.bfloat16`                   


INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]


...

WARN  Module Tree AutoCompat: Final module_tree: ['model', 'layers', '#', {'feed_forward': ('w1', 'w3', 'w2')}]


INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 1,
  "eos_token_id": 7,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 0,
  "use_cache": true
}



INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.


INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 1,
  "do_sample": true,
  "eos_token_id": 7,
  "pad_token_id": 0
}



INFO  Kernel: loaded -> `[]`                                                   


In [ ]:
# load calibration dataset
from datasets import load_dataset
data = load_dataset("mikasenghaas/wikitext-2", split="test").select(range(512))

README.md:   0%|          | 0.00/493 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.20M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/641k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/713k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17556 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1841 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2183 [00:00<?, ? examples/s]

In [ ]:
model.quantize(data, batch_size=2)

INFO  Packing Kernel: selected: `TorchLinear`                                  


INFO  Packing Kernel: selected: `TorchLinear`                                  


WARN  Quantize: 52 input_ids with length <= 10 were removed. Use quantize(calibration_data_min_length=10) to set a custom minimum length.


INFO  Calibration: Sort in descending order by length                          


INFO  Calibration: Total padded tokens: 271                                    


...

INFO  +-------------------+-------+--------+--------+---------+--------+--------------------------------+


INFO  | Capture inputs    | 1     | 4.394  | 4.394  | 4.394   | 100.0% | cache_inputs:Lfm2DecoderLayer  |


INFO  +-------------------+-------+--------+--------+---------+--------+--------------------------------+


Quantizing layer 2 of 15            [2 of 15] ▍░| 0:00:39 / 0:03:28 [3/16] 18.8%
Layer 0 Submodule finalize 0/3 Waiting for completions... | 0:00:30 / 0:01:30 [0
Compiling extension: pack_block_cpu... elapsed 30s / estimated ~31s  | 0:00:30 /
Layer 1 Submodule finalize 0/3 Waiting for completions... | 0:00:18 / 0:00:54 [0
Quantizing layer 0 of 15 [0 of 15] ▉░░░░░░░░░░░░░| 0:00:00 / 0:00:00 [1/16] 6.2%

ValueError: layer module item `feed_forward.w1` not found in model, please check your model config.

# 10-Min Exercise: GGUF Model + VRAM Estimation

* **Step 0 — Check GPU (IMPORTANT)**

In [6]:
! nvidia-smi

Sat May 16 08:26:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   78C    P0             34W /   70W |    8587MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

*  **Step 1 — Free up GPU (IMPORTANT)**

In [12]:
import torch
import gc

# 1. Clear variables from Python memory (if you have specific large tensors/models)
del model

# 2. Force Python to collect garbage right now
gc.collect()

# 3. Clear PyTorch's internal GPU memory caching allocator
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

 * **Step 2 — Fill up next table**

estimate memory using: ***Example: 7B model***

| Format | Bytes / Param | Estimated Memory |
| ------ | ------------- | ---------------- |
| FP16   | 2 bytes       | 14 GB                |
| INT8   | 1 byte        | 7 GB                |
| Q4     | 0.5 byte      | 3.5 GB               |

* **step3 - Load GGUF Model using llama.cpp and test with an simple input**

In [3]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="TheBloke/Mistral-7B-Instruct-v0.1-GGUF",  # example model
    filename="mistral-7b-instruct-v0.1.Q4_K_M.gguf",   # Q4 fits in Colab VRAM
    local_dir="./models"
)

print("Model downloaded to:", model_path)

mistral-7b-instruct-v0.1.Q4_K_M.gguf:   0%|          | 0.00/4.37G [00:00<?, ?B/s]

Model downloaded to: models/mistral-7b-instruct-v0.1.Q4_K_M.gguf


In [7]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,   # offload all layers to GPU
    n_ctx=2048,
    verbose=True       # shows layer offloading info
)

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB
llama_model_loader: loaded meta data with 20 key-value pairs and 291 tensors from models/mistral-7b-instruct-v0.1.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.1
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model

What is the capital of France?


In [9]:
response = llm.create_chat_completion(
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ],
    max_tokens=128
)

print(response["choices"][0]["message"]["content"])

Llama.generate: 1 prefix-match hit, remaining 17 prompt tokens to eval
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
llama_perf_context_print:        load time =     344.23 ms
llama_perf_context_print: prompt eval time =     509.34 ms /    17 tokens (   29.96 ms per token,    33.38 tokens per second)
llama_perf_context_print:        eval time =     234.52 ms /     7 runs   (   33.50 ms per token,    29.85 tokens per second)
llama_perf_context_print:       total time =     747.11 ms /    24 tokens
llama_perf_context_print:    graphs reused =          6


 The capital of France is Paris.


**👏🏼 i wish you learned something, and happy hacking :)**